# Forecasting de Ventas 2025

Este notebook importa las librerías necesarias y carga el archivo de inferencia para realizar análisis y predicciones sobre las ventas de 2025.

## 1. Importar librerías necesarias
Importamos todas las librerías utilizadas en el notebook de entrenamiento para el procesamiento y análisis de datos.

In [646]:
# Importación de librerías principales
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import holidays
from sklearn import datasets, model_selection, metrics, preprocessing, linear_model, ensemble, tree, neighbors, svm, cluster, decomposition

## 2. Cargar archivo de inferencia en DataFrame
Cargamos el archivo 'ventas_2025_inferencia.csv' ubicado en 'data/raw/inferencia' en un DataFrame llamado inferencia_df.

In [647]:
# Cargar el archivo de inferencia en un DataFrame
inferencia_path = '../data/raw/inferencia/ventas_2025_inferencia.csv'
inferencia_df = pd.read_csv(inferencia_path)

# Mostrar las primeras filas para verificación
display(inferencia_df.head())

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,Amazon,Decathlon,Deporvillage
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,89.51,113.43,104.78
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,128.73,112.91,122.88
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,84.28,74.51,85.57
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,75.54,70.32,71.13
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,33.84,31.32,34.41


## Preparación de inferencia_df para el modelo de forecasting
Aplicamos exactamente las mismas transformaciones, ingeniería de variables y codificaciones que en el notebook de entrenamiento para dejar inferencia_df listo para la inferencia.

In [648]:
# 1. Conversión de columna fecha a datetime
inferencia_df['fecha'] = pd.to_datetime(inferencia_df['fecha'])

# 2. Crear variables temporales y de calendario
meses_es = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
orden_dias_es = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
inferencia_df['dia_semana_es'] = inferencia_df['fecha'].dt.day_name(locale='es_ES')
inferencia_df['dia_semana_es'] = pd.Categorical(inferencia_df['dia_semana_es'], categories=orden_dias_es, ordered=True)

try:
    inferencia_df['mes_nombre'] = inferencia_df['fecha'].dt.month.apply(lambda x: meses_es[x-1])
except:
    inferencia_df['mes_nombre'] = inferencia_df['mes']


inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day

inferencia_df['nombre_dia_semana'] = inferencia_df['fecha'].dt.day_name(locale='es_ES')
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5,6])
spain_holidays = holidays.country_holidays('ES', years=inferencia_df['año'].unique())
inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(spain_holidays)

def es_black_friday(fecha):
    if fecha.month == 11:
        ultimo_viernes = max([d for d in pd.date_range(start=fecha.replace(day=1), end=fecha.replace(day=30)) if d.weekday() == 4])
        return fecha == ultimo_viernes
    return False
inferencia_df['es_Black_Friday'] = inferencia_df['fecha'].apply(es_black_friday)

def es_cyber_monday(fecha):
    if fecha.month == 11:
        ultimo_viernes = max([d for d in pd.date_range(start=fecha.replace(day=1), end=fecha.replace(day=30)) if d.weekday() == 4])
        cyber_monday = ultimo_viernes + pd.Timedelta(days=3)
        return fecha == cyber_monday
    return False
inferencia_df['es_Cyber_Monday'] = inferencia_df['fecha'].apply(es_cyber_monday)

inferencia_df['semana_año'] = inferencia_df['fecha'].dt.isocalendar().week
inferencia_df['dia_año'] = inferencia_df['fecha'].dt.dayofyear
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter

C:\Users\didac\AppData\Local\Temp\ipykernel_49348\3778906677.py:25: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  inferencia_df['es_festivo'] = inferencia_df['fecha'].isin(spain_holidays)


In [649]:
# 3. Crear variables LAG y media móvil de 7 días para unidades_vendidas (por año)
lags = range(1, 8)
for lag in lags:
    inferencia_df[f'unidades_vendidas_lag{lag}'] = inferencia_df.groupby(['año'])['unidades_vendidas'].shift(lag)
inferencia_df['unidades_vendidas_mm7'] = inferencia_df.groupby(['año'])['unidades_vendidas'].transform(lambda x: x.rolling(window=7, min_periods=7).mean())

In [650]:
inferencia_df

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,dia_año,trimestre,unidades_vendidas_lag1,unidades_vendidas_lag2,unidades_vendidas_lag3,unidades_vendidas_lag4,unidades_vendidas_lag5,unidades_vendidas_lag6,unidades_vendidas_lag7,unidades_vendidas_mm7
0,2025-10-25,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,26.0,113.13,2941.38,...,298,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-10-25,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,27.0,141.89,3831.03,...,298,4,26.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-10-25,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,5.0,85.79,428.95,...,298,4,27.0,26.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-10-25,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,3.0,76.19,228.57,...,298,4,5.0,27.0,26.0,NaN,NaN,NaN,NaN,NaN
4,2025-10-25,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,3.0,35.48,106.44,...,298,4,3.0,5.0,27.0,26.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2025-11-30,PROD_020,Quechua MH500,Outdoor,Ropa Montaña,80,False,NaN,79.64,NaN,...,334,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
884,2025-11-30,PROD_021,Manduka PRO Yoga Mat,Wellness,Esterilla Yoga,130,True,NaN,130.00,NaN,...,334,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
885,2025-11-30,PROD_022,Gaiam Premium Yoga Block,Wellness,Bloque Yoga,20,False,NaN,20.18,NaN,...,334,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
886,2025-11-30,PROD_023,Liforme Yoga Pad,Wellness,Rodillera Yoga,35,False,NaN,34.79,NaN,...,334,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [651]:
primeras_20 = inferencia_df.iloc[165:170, 10:20]
print(primeras_20.head())  # Muestra las primeras filas de esas columnas

     Amazon  Decathlon  Deporvillage   año  mes  dia_semana dia_semana_es  \
165   19.09      18.90         18.94  2025   10           4       Viernes   
166   33.35      30.61         34.72  2025   10           4       Viernes   
167   47.99      42.83         49.58  2025   10           4       Viernes   
168   81.78     117.08        102.97  2025   11           5        Sábado   
169  121.00     116.13        120.11  2025   11           5        Sábado   

    mes_nombre  dia_mes nombre_dia_semana  
165    Octubre       31           Viernes  
166    Octubre       31           Viernes  
167    Octubre       31           Viernes  
168  Noviembre        1            Sábado  
169  Noviembre        1            Sábado  


In [652]:
# 4. Crear variable descuento_porcentaje
des = (inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base'] * 100
inferencia_df['descuento_porcentaje'] = des

# 5. Crear variable precio_competencia como la media de Amazon, Decathlon y Deporvillage
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['precio_competencia'] = inferencia_df[competidores].mean(axis=1)

# 6. Crear variable ratio_precio: nuestro precio_venta / precio_competencia
inferencia_df['ratio_precio'] = inferencia_df['precio_venta'] / inferencia_df['precio_competencia']

# 7. Eliminar columnas de Amazon, Decathlon y Deporvillage
inferencia_df = inferencia_df.drop(columns=competidores)

In [653]:
inferencia_df.shape

(888, 35)

In [654]:
# 8. Crear copia de variables con sufijo _h para one-hot encoding
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

# 9. One hot encoding sobre las variables *_h
inferencia_df = pd.get_dummies(inferencia_df, columns=['nombre_h', 'categoria_h', 'subcategoria_h'], drop_first=False)

In [655]:
inferencia_df.shape

(888, 79)

In [656]:
# 10. En inferencia, no eliminamos registros con nulos en LAGs y media móvil
# Solo eliminamos los registros de octubre y dejamos los de noviembre
del_octubre = inferencia_df['mes'] != 11
inferencia_df = inferencia_df[~del_octubre].copy()

In [657]:
inferencia_df.shape

(720, 79)

In [658]:
primeras_20 = inferencia_df.iloc[100:120, 10:20]
print(primeras_20.head())  # Muestra las primeras filas de esas columnas

      año  mes  dia_semana dia_semana_es mes_nombre  dia_mes  \
268  2025   11           2     Miércoles  Noviembre        5   
269  2025   11           2     Miércoles  Noviembre        5   
270  2025   11           2     Miércoles  Noviembre        5   
271  2025   11           2     Miércoles  Noviembre        5   
272  2025   11           2     Miércoles  Noviembre        5   

    nombre_dia_semana  es_fin_semana  es_festivo  es_Black_Friday  
268         Miércoles          False       False            False  
269         Miércoles          False       False            False  
270         Miércoles          False       False            False  
271         Miércoles          False       False            False  
272         Miércoles          False       False            False  


In [659]:
inferencia_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 720 entries, 168 to 887
Data columns (total 79 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   fecha                                      720 non-null    datetime64[ns]
 1   producto_id                                720 non-null    object        
 2   nombre                                     720 non-null    object        
 3   categoria                                  720 non-null    object        
 4   subcategoria                               720 non-null    object        
 5   precio_base                                720 non-null    int64         
 6   es_estrella                                720 non-null    bool          
 7   unidades_vendidas                          0 non-null      float64       
 8   precio_venta                               720 non-null    float64       
 9   ingresos                

In [660]:
# 11. Eliminar registros de octubre y dejar solo los de noviembre
inferencia_df = inferencia_df[inferencia_df['mes'] == 11].copy()
inferencia_df

,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,False,False,False,False,False,False,True,False
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,False,False,False,False,False,False,True,False
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,False,False,False,False,False,False,True,False
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2025-11-30,PROD_020,Quechua MH500,Outdoor,Ropa Montaña,80,False,NaN,79.64,NaN,...,False,False,False,False,False,False,True,False,False,False
884,2025-11-30,PROD_021,Manduka PRO Yoga Mat,Wellness,Esterilla Yoga,130,True,NaN,130.00,NaN,...,True,False,False,False,False,False,False,False,False,False
885,2025-11-30,PROD_022,Gaiam Premium Yoga Block,Wellness,Bloque Yoga,20,False,NaN,20.18,NaN,...,False,False,False,False,False,False,False,False,False,False
886,2025-11-30,PROD_023,Liforme Yoga Pad,Wellness,Rodillera Yoga,35,False,NaN,34.79,NaN,...,False,False,False,False,False,True,False,False,False,False


In [661]:
# 12. Guardar el DataFrame transformado
guardar_path = '../data/processed/inferencia_df_transformado.csv'
inferencia_df.to_csv(guardar_path, index=False)
print(f'DataFrame de inferencia transformado guardado en {guardar_path}')

DataFrame de inferencia transformado guardado en ../data/processed/inferencia_df_transformado.csv


In [662]:
inferencia_df.head()


,fecha,producto_id,nombre,categoria,subcategoria,precio_base,es_estrella,unidades_vendidas,precio_venta,ingresos,...,subcategoria_h_Esterilla Yoga,subcategoria_h_Mancuernas Ajustables,subcategoria_h_Mochila Trekking,subcategoria_h_Pesa Rusa,subcategoria_h_Pesas Casa,subcategoria_h_Rodillera Yoga,subcategoria_h_Ropa Montaña,subcategoria_h_Ropa Running,subcategoria_h_Zapatillas Running,subcategoria_h_Zapatillas Trail
168,2025-11-01,PROD_001,Nike Air Zoom Pegasus 40,Running,Zapatillas Running,115,True,NaN,115.00,NaN,...,False,False,False,False,False,False,False,False,True,False
169,2025-11-01,PROD_002,Adidas Ultraboost 23,Running,Zapatillas Running,135,True,NaN,135.00,NaN,...,False,False,False,False,False,False,False,False,True,False
170,2025-11-01,PROD_003,Asics Gel Nimbus 25,Running,Zapatillas Running,85,False,NaN,86.39,NaN,...,False,False,False,False,False,False,False,False,True,False
171,2025-11-01,PROD_004,New Balance Fresh Foam X 1080v12,Running,Zapatillas Running,75,False,NaN,74.09,NaN,...,False,False,False,False,False,False,False,False,True,False
172,2025-11-01,PROD_005,Nike Dri-FIT Miler,Running,Ropa Running,35,False,NaN,34.76,NaN,...,False,False,False,False,False,False,False,True,False,False


In [663]:
inferencia_df.columns 

Index(['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria',
       'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta',
       'ingresos', 'año', 'mes', 'dia_semana', 'dia_semana_es', 'mes_nombre',
       'dia_mes', 'nombre_dia_semana', 'es_fin_semana', 'es_festivo',
       'es_Black_Friday', 'es_Cyber_Monday', 'semana_año', 'dia_año',
       'trimestre', 'unidades_vendidas_lag1', 'unidades_vendidas_lag2',
       'unidades_vendidas_lag3', 'unidades_vendidas_lag4',
       'unidades_vendidas_lag5', 'unidades_vendidas_lag6',
       'unidades_vendidas_lag7', 'unidades_vendidas_mm7',
       'descuento_porcentaje', 'precio_competencia', 'ratio_precio',
       'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23',
       'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552',
       'nombre_h_Columbia Silver Ridge',
       'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900',
       'nombre_h_Domyos Kit Mancuernas 20kg',
       